# Backfill: Google Sheet Sales → daily_sales_v2

One-time import of 2025 Google Sheet daily sales into `daily_sales_v2`.

**Dedup strategy:** Only inserts rows for date+store combos that have **no existing data** in `daily_sales_v2`.  
API data always takes priority — GS data fills gaps only.

**Steps:**
1. Review and complete the store name mapping below
2. Upload your GS CSV to Databricks (Catalog → Files, or DBFS)
3. Run All

In [ ]:
# ── Config ────────────────────────────────────────────────────────

GS_CSV_PATH = "/Workspace/Users/davidgao734@gmail.com/boba-cafe/POS/data/2025_google_sheet.csv"

from pipeline.config import DAILY_SALES_TABLE

In [ ]:
# ── Store Name Mapping ────────────────────────────────────────────
# Maps Google Sheet store names → standard daily_sales_v2 store names.
# Stores set to None are kept with their original GS name (closed/retired stores).

GS_STORE_MAP = {
    # Active stores
    "Боба Рэббит":               "НОВО КП",
    "БОБА РЭББИТ БОН ПАССАЖ":   "НОВО КП",
    "БОБА РЭББИТ ЧМ":            "ЧЕРНОМОРСКИЙ",
    "Советов":                   "СОВЕТОВ",
    "Черноморский":              "ЧЕРНОМОРСКИЙ",
    "Грин Парк":                 "ГРИН ПАРК",
    "Галерея Краснодар":         "ГАЛЕРЕЯ",
    "Краснодар Оз Мол":          "ОЗМОЛЛ",
    "КРАСНОДАР КРАСНАЯ ПЛОЩАДЬ": "КПК",
    "Красная":                   "БОН ПАССАЖ",

    # Unidentified terminals
    "КРАСНОДАР ВОЛОДЯ":          "UNKNOWN_КРВОЛ",
    "КРАСНОДАР запасная касса":  "UNKNOWN_КРЗАП",
    "БОБА РЭББИТ ЗАПАС":         "UNKNOWN_ББЗАП",
    "БОБА РЭББИТ ВОЛОДЯ":        "UNKNOWN_ББВОЛ",

    # Closed/retired stores — kept with original names
    "Анапа":                     None,
    "Анапа Черноморская":        None,
    "Геленджик":                 None,
    "Коса":                      None,
    "Лента":                     None,
    "Основной офис":             None,
}

In [ ]:
# ── Load & Normalize Google Sheet Data ───────────────────────────

import pandas as pd

gs = pd.read_csv(GS_CSV_PATH)

# Rename columns to match daily_sales_v2 schema
gs = gs.rename(columns={"cash": "payment_type", "total": "revenue"})

# Map payment types: non-cash → card (online was not tracked separately in GS)
gs["payment_type"] = gs["payment_type"].map({"cash": "cash", "non-cash": "card"})

# Apply store name mapping
def map_store(name):
    if name in GS_STORE_MAP:
        mapped = GS_STORE_MAP[name]
        return mapped if mapped is not None else name
    return name  # Keep as-is if not in map

gs["store"] = gs["store"].apply(map_store)
gs["date"] = pd.to_datetime(gs["date"]).dt.date
gs["revenue"] = gs["revenue"].fillna(0).astype(int)

print(f"Loaded {len(gs):,} rows from Google Sheet CSV")
gs.head(10)

In [ ]:
# ── Load Existing Delta Table ─────────────────────────────────────

from pipeline.transforms import _get_spark

spark = _get_spark()

existing = spark.table(DAILY_SALES_TABLE).toPandas()
existing["date"] = existing["date"].astype(str)

# Only treat a date+store as "taken" if it has real revenue (not just zero-fill rows)
has_revenue = existing.groupby(["date", "store"])["revenue"].sum().reset_index()
has_revenue = has_revenue[has_revenue["revenue"] > 0]

existing_keys = set(zip(has_revenue["date"], has_revenue["store"]))

print(f"Existing Delta table: {len(existing):,} rows")
print(f"Date+store combos with real revenue: {len(existing_keys):,}")

In [ ]:
# ── Filter to Gaps Only ───────────────────────────────────────────

# Only keep GS rows where the date+store combo has NO existing API data
gs["_key"] = list(zip(gs["date"].astype(str), gs["store"]))
to_insert = gs[~gs["_key"].isin(existing_keys)].drop(columns="_key")

print(f"GS rows to insert (gaps only): {len(to_insert):,}")
print(f"GS rows skipped (already have API data): {len(gs) - len(to_insert):,}")
print()
print("Stores being backfilled:")
print(to_insert.groupby("store")["date"].agg(["min", "max", "count"]).to_string())

In [ ]:
# ── Add missing online=0 rows for GS stores ──────────────────────
# GS data has no online column — add online=0 rows so schema is complete

gs_dates_stores = to_insert[["date", "store"]].drop_duplicates()
online_rows = gs_dates_stores.copy()
online_rows["payment_type"] = "online"
online_rows["revenue"] = 0

# Only add online rows where they don't already exist
existing_online_keys = set(
    zip(
        existing[existing["payment_type"] == "online"]["date"].astype(str),
        existing[existing["payment_type"] == "online"]["store"]
    )
)
online_rows["_key"] = list(zip(online_rows["date"].astype(str), online_rows["store"]))
online_rows = online_rows[~online_rows["_key"].isin(existing_online_keys)].drop(columns="_key")

to_insert = pd.concat([to_insert, online_rows], ignore_index=True)
to_insert["date"] = pd.to_datetime(to_insert["date"]).dt.date

print(f"Total rows to insert (including online=0): {len(to_insert):,}")
to_insert.sample(10)

In [ ]:
# ── Write to Delta Table ─────────────────────────────────────────

from pipeline.transforms import DAILY_SALES_SCHEMA

sdf = spark.createDataFrame(to_insert, schema=DAILY_SALES_SCHEMA)
sdf.write.mode("append").saveAsTable(DAILY_SALES_TABLE)

print(f"Done. Inserted {len(to_insert):,} rows into {DAILY_SALES_TABLE}")